In [1]:
#exécuter puis redémarrer le kernel
!pip uninstall -y pyOpenSSL
!pip install -U "cryptography>=43,<47"
!pip check

pyspark 3.5.0 requires py4j, which is not installed.
conda 23.9.0 requires pyopenssl, which is not installed.
certipy 0.1.3 requires pyopenssl, which is not installed.


In [2]:
!pip -q install mlflow scikit-learn pandas numpy

## 1 - Configuration Tracking (backend SQLite + experiment)

In [3]:
import os, pathlib, tempfile
import mlflow

# Backend SQLite local (robuste)
db_path = pathlib.Path(tempfile.gettempdir()) / "mlflow_demo.db"
tracking_uri = f"sqlite:///{db_path}"

mlflow.set_tracking_uri(tracking_uri)

experiment_name = "demo-mlflow-notebook"
mlflow.set_experiment(experiment_name)

# Pour que `!mlflow run` log au même endroit que Python
os.environ["MLFLOW_TRACKING_URI"] = tracking_uri
os.environ["MLFLOW_EXPERIMENT_NAME"] = experiment_name

print("Tracking URI =", tracking_uri)
print("Experiment  =", experiment_name)

2025/12/16 20:39:56 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/16 20:39:56 INFO mlflow.store.db.utils: Updating database tables
2025/12/16 20:39:56 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/16 20:39:56 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/16 20:39:56 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/16 20:39:56 INFO alembic.runtime.migration: Will assume non-transactional DDL.


Tracking URI = sqlite:////tmp/mlflow_demo.db
Experiment  = demo-mlflow-notebook


## 2 - Préparer les données (Iris)

In [4]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((112, 4), (38, 4))

# === MLflow Trackings ===

## 3 - TRACKING : entraîner + loguer params / metrics / artifact / model

In [5]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import mlflow
import mlflow.sklearn

params = {"n_estimators": 200, "max_depth": 10, "random_state": 42}

with mlflow.start_run(run_name="tracking-rf") as run:
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)

    acc = accuracy_score(y_test, model.predict(X_test))

    mlflow.log_params(params)
    mlflow.log_metric("accuracy", acc)

    fi = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    fi.to_csv("feature_importances.csv")
    mlflow.log_artifact("feature_importances.csv")

    mlflow.sklearn.log_model(model, artifact_path="model")

print("run_id =", run.info.run_id)
print("accuracy =", acc)

2025/12/16 20:39:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


run_id = edd16fe29ff0434f9aec608d34e26c6a
accuracy = 0.8947368421052632


## 4 - TRACKING : lister les runs dans l’experiment

In [6]:
import mlflow

exp = mlflow.get_experiment_by_name(os.environ["MLFLOW_EXPERIMENT_NAME"])
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=["start_time DESC"])
runs[["run_id", "status", "metrics.accuracy"]].head(10)

,run_id,status,metrics.accuracy
0,edd16fe29ff0434f9aec608d34e26c6a,FINISHED,0.894737
1,252256ceb3844e22b9f81ccd63d7e1a0,FINISHED,0.921053
2,5fb6ccd815064ea39d4d4da0a71e17d9,FINISHED,0.921053
3,2b947f5c1b684bc7abb63268d5a533f3,FINISHED,0.894737
4,cbc6ae9659f54b5c9c545d69ddaeb48e,FINISHED,0.894737
5,97167f68dd884e9dacff10ccc95471ee,FINISHED,0.894737
6,6901c153e9764e9d97b6c2e284c37a95,FINISHED,0.894737
7,1812439198184cd88a83099e5f61d5c3,FINISHED,0.894737
8,e118f08703eb4667b534640b3d63016e,FINISHED,0.894737


# === MLflow Projects ===

## 5 - Créer un dossier “Project” (MLproject + train.py)

In [7]:
import textwrap, pathlib, tempfile

project_dir = pathlib.Path(tempfile.gettempdir()) / "mlflow_project_demo_notebook"

(project_dir / "train.py").write_text(textwrap.dedent("""\
import argparse
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

def main(n_estimators: int, max_depth: int):
    # IMPORTANT (MLflow Projects):
    # - mlflow run crée déjà un run et fournit MLFLOW_RUN_ID / MLFLOW_EXPERIMENT_ID via env
    # - On ne relance pas mlflow.start_run() ici
    iris = load_iris(as_frame=True)
    X = iris.data
    y = iris.target

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    params = {"n_estimators": n_estimators, "max_depth": max_depth, "random_state": 42}

    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))

    mlflow.log_params(params)
    mlflow.log_metric("accuracy", acc)

    fi = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
    fi.to_csv("feature_importances.csv")
    mlflow.log_artifact("feature_importances.csv")

    mlflow.sklearn.log_model(model, artifact_path="model")

if __name__ == "__main__":
    p = argparse.ArgumentParser()
    p.add_argument("--n_estimators", type=int, default=200)
    p.add_argument("--max_depth", type=int, default=5)
    args = p.parse_args()
    main(args.n_estimators, args.max_depth)
"""))

print("train.py updated:", project_dir / "train.py")

train.py updated: /tmp/mlflow_project_demo_notebook/train.py


## 6 - PROJECTS : exécuter mlflow run

In [8]:
import os

os.environ.pop("MLFLOW_EXPERIMENT_NAME", None)
os.environ.pop("MLFLOW_EXPERIMENT_ID", None)

project_path = str(project_dir)
!mlflow run "{project_path}" -P n_estimators=300 -P max_depth=4

2025/12/16 20:40:01 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/16 20:40:01 INFO mlflow.store.db.utils: Updating database tables
2025/12/16 20:40:01 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/16 20:40:01 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/16 20:40:01 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/16 20:40:01 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/16 20:40:02 INFO mlflow.utils.conda: Conda environment mlflow-36511fe3c375161bd3c56e97dcfc620e0d8eb52d already exists.
2025/12/16 20:40:02 INFO mlflow.projects.utils: === Created directory /tmp/tmp9bwnzhko for downloading remote URIs passed to arguments of type 'path' ===
2025/12/16 20:40:02 INFO mlflow.projects.backend.local: === Running command 'source activate mlflow-36511fe3c375161bd3c56e97dcfc620e0d8eb52d 1>&2 && python train.py --n_estimators 300 --max_depth 4' in run with ID 'ce22a5c28c2d

# === MLflow Models ===

## MODELS : retrouver le dernier run et charger le modèle

In [9]:
import mlflow, mlflow.pyfunc

exp = mlflow.get_experiment_by_name("demo-mlflow-notebook")
last = mlflow.search_runs([exp.experiment_id], order_by=["start_time DESC"], max_results=1)
run_id = last.iloc[0]["run_id"]

mlflow.pyfunc.load_model(f"runs:/{run_id}/model")

mlflow.pyfunc.loaded_model:
  artifact_path: /home/jovyan/notebooks/mlruns/1/models/m-d75b3ed33a884671a6f4830742f8efed/artifacts
  flavor: mlflow.sklearn
  run_id: edd16fe29ff0434f9aec608d34e26c6a

In [10]:
import mlflow, mlflow.pyfunc

exp = mlflow.get_experiment_by_name("demo-mlflow-notebook")
last = mlflow.search_runs([exp.experiment_id], order_by=["start_time DESC"], max_results=1)
run_id = last.iloc[0]["run_id"]

model = mlflow.pyfunc.load_model(f"runs:/{run_id}/model")
print("Loaded model from run_id =", "812439198184cd88a83099e5f61d5c3") #run_id)

Loaded model from run_id = 812439198184cd88a83099e5f61d5c3


In [11]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
X = iris.data

pred = model.predict(X.tail(20))
pred


array([2, 2, 2, 2, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2])

In [12]:
from sklearn.metrics import accuracy_score

y_true = iris.target
y_pred = model.predict(X)

accuracy_score(y_true, y_pred)

0.9733333333333334

In [13]:
print(f'mlflow models serve -m "runs:/{run_id}/model" -p 5001 --no-conda')

mlflow models serve -m "runs:/edd16fe29ff0434f9aec608d34e26c6a/model" -p 5001 --no-conda
